# Demo: Proof Reconstructor

This notebook demonstrates the current explanation-reconstruction slice in `rdflib-reasoning-engine`.
It rebuilds a `DirectProof` from engine-native `DerivationRecord` values using `DerivationProofReconstructor`.

Current scope:
- triple goals only
- recursive reconstruction from derivation logs
- `RuleApplication` nodes for explained conclusions
- `ProofLeaf` fallback for unexplained premises or goals

In [1]:
from pprint import pprint

from rdflib import BNode, URIRef
from rdflib.namespace import RDF, RDFS
from rdflib_reasoning.engine import (
    DerivationProofReconstructor,
    DerivationRecord,
    RuleId,
    TripleFact,
)

In [2]:
context = BNode()

alice = URIRef("urn:test:alice")
human = URIRef("urn:test:Human")
mammal = URIRef("urn:test:Mammal")
animal = URIRef("urn:test:Animal")

goal = TripleFact(context=context, triple=(alice, RDF.type, animal))
intermediate = TripleFact(context=context, triple=(alice, RDF.type, mammal))
premise_a = TripleFact(context=context, triple=(alice, RDF.type, human))
premise_b = TripleFact(context=context, triple=(human, RDFS.subClassOf, mammal))
premise_c = TripleFact(context=context, triple=(mammal, RDFS.subClassOf, animal))

records = [
    DerivationRecord(
        context=context,
        conclusions=[intermediate],
        premises=[premise_a, premise_b],
        rule_id=RuleId(ruleset="rdfs", rule_id="rdfs9"),
        depth=1,
    ),
    DerivationRecord(
        context=context,
        conclusions=[goal],
        premises=[intermediate, premise_c],
        rule_id=RuleId(ruleset="rdfs", rule_id="rdfs9"),
        depth=2,
    ),
]

proof = DerivationProofReconstructor().reconstruct(goal, records)
proof

DirectProof(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), goal=TripleFact(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Animal'))), proof=RuleApplication(node_kind='rule_application', conclusions=[TripleFact(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Animal')))], premises=[RuleApplication(node_kind='rule_application', conclusions=[TripleFact(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Mammal')))], premises=[ProofLeaf(node_kind='leaf', c

In [3]:
print("Verdict:", proof.verdict)
print("Top-level node kind:", proof.proof.node_kind)
print(
    "Nested premise node kinds:",
    [premise.node_kind for premise in proof.proof.premises],
)

pprint(proof.model_dump(mode="python"))

Verdict: proved
Top-level node kind: rule_application
Nested premise node kinds: ['rule_application', 'leaf']
{'context': rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'),
 'goal': {'context': rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'),
          'kind': 'triple',
          'triple': (rdflib.term.URIRef('urn:test:alice'),
                     rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                     rdflib.term.URIRef('urn:test:Animal'))},
 'notes': None,
 'proof': {'conclusions': [{'context': rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'),
                            'kind': 'triple',
                            'triple': (rdflib.term.URIRef('urn:test:alice'),
                                       rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                                       rdflib.term.URIRef('urn:test:Animal'))}],
           'derivation': {'bindings': [],
                          'conclusion

In [4]:
# If no derivation record exists for the goal, reconstruction falls back to a leaf.
unexplained_goal = TripleFact(
    context=context,
    triple=(URIRef("urn:test:bob"), RDF.type, URIRef("urn:test:Human")),
)

unexplained = DerivationProofReconstructor().reconstruct(unexplained_goal, records)
print("Verdict:", unexplained.verdict)
print("Proof node kind:", unexplained.proof.node_kind)
unexplained

Verdict: incomplete
Proof node kind: leaf


DirectProof(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), goal=TripleFact(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), kind='triple', triple=(rdflib.term.URIRef('urn:test:bob'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Human'))), proof=ProofLeaf(node_kind='leaf', claim=TripleFact(context=rdflib.term.BNode('N0653e8fbdff24ab4b48a0d2d2763d208'), kind='triple', triple=(rdflib.term.URIRef('urn:test:bob'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Human'))), grounding=[]), verdict='incomplete', notes=None)